---
---
---
<a name="FSDP2"></a>
## B) Make `QLoRA` work with `FSDP2` [Difficulty: Medium to Hard] [Max points: 10]

1. Goal: Write a single Python script to finetune Llama 3.1 8B on 2x or more GPUs with FSDP2.

2. You must showcase this working in a free **Kaggle notebook with 2 x Tesla T4 GPUs**.

3. Pipeline parallelism is also fine, but must utilize [`zero bubble scheduling`](https://pytorch.org/docs/stable/distributed.pipelining.html#torch.distributed.pipelining.schedules.ScheduleInterleavedZeroBubble) somehow.

4. Can use a pre-quantized 4bit BnB safetensor file from [Unsloth's HF page](https://huggingface.co/unsloth) or a full 16bit one, but must do QLoRA.

5. Can use `accelerate` but must be FSDP2 or related - you can investigate https://github.com/huggingface/accelerate/pull/3394, Torch Titan, other repos etc.

6. Must be fully `transformers` compatible - so we must use `TrainingArguments` and `Trainer`, or `TRL` related classes.

7. The loss must be equivalent to single GPU training.

8. You must enable all features in FSDP2 - ie showcase offloading, checkpointing, mixed precision training etc.

9. You can use `nf4` from `torch AO`, but best from `bitsandbytes`.

10. Finally showcase everything working in a free Kaggle 2x Tesla T4 notebook.

## Marking Criteria for B) Max points = 10
```python
if attemped_B:
    B_score = 0
    if FSDP2_works_with_QLoRA:
        if torch_compile_works: B_score += 5
        else: B_score += 3
        if uses_part_A_and_single_kernel_and_faster: B_score += 3
        elif uses_torchAO:
            if torchAO_slower_than_BnB: B_score -= 3
    elif TP_or_PP_with_QLoRA:
        if zero_bubble: B_score += 3
        else: B_score += 2
    elif FSDP1_works_with_QLoRA:
        B_score += 1
    if kaggle_notebook_2_tesla_t4_example:
        B_score += 2
    else:
        B_score = 0
    final_score += B_score
else:
    final_score -= 2
```

In [1]:
# # Code to install Unsloth, Triton, Torch etc
# %%capture
# !pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton
# !pip install --no-deps cut_cross_entropy unsloth_zoo
# !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
# !pip install --no-deps unsloth

In [2]:
import torch

torch.__version__

'2.6.0+cu126'

In [3]:
# Helpful functions used through the entire notebook
import torch
import torch.nn as nn
from transformers import set_seed
import time
import inspect
import os

major_version, minor_version = torch.cuda.get_device_capability()
HAS_BFLOAT16 = major_version >= 8
from inspect import currentframe as _C, getframeinfo

_F = lambda c: getframeinfo(c).lineno  # Gets line number
WARN = lambda x: print(f"\033[31m{x}\033[0m")  # Red colored warnings


# https://stackoverflow.com/questions/18425225/getting-the-name-of-a-variable-as-a-string
def NAME(var):
    callers_local_vars = inspect.currentframe().f_back.f_locals.items()
    names = [var_name for var_name, var_val in callers_local_vars if var_val is var]
    return names[0] if len(names) != 0 else ""


def assert_same(x, y, line, dtype):
    assert x.dtype == dtype
    try:
        torch.testing.assert_close(x, y, check_stride=True)
    except Exception as error:
        raise RuntimeError(
            f"Failed allclose at line [{line}]: {NAME(x)}, {NAME(y)}\n{str(error)}"
        )


os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [4]:
# HELPFUL functions to undo Unsloth patches:
import sys


def remove_patched_module(package_name):
    modules_to_delete = [
        name
        for name in sys.modules
        if name == package_name or name.startswith(package_name + ".")
    ]
    for name in modules_to_delete:
        del sys.modules[name]


remove_patched_module("trl")
remove_patched_module("transformers")
remove_patched_module("peft")
remove_patched_module("bitsandbytes")

In [5]:
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True," "roundup_power2_divisions:[32:256,64:128,256:64,>:32]"
)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType

max_seq_length = 2048
torch.set_default_dtype(torch.float16)
model_name = "unsloth/meta-Llama-3.1-8B-Instruct-bnb-4bit"
dtype = torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # device_map="auto",
    attn_implementation="sdpa",
    quantization_config=bnb_config,
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "right"

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Get LoRA and setup model
model = get_peft_model(model, lora_config)
with torch.no_grad():
    for name, param in model.named_parameters():
        if ".lora_A." in name or ".lora_B." in name:
            param.requires_grad_(True)
        else:
            param.requires_grad_(False)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# Get dataset
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

url = "https://huggingface.co/datasets/laion/OIG/resolve/main/unified_chip2.jsonl"
dataset = load_dataset("json", data_files={"train": url}, split="train[:10%]")

/opt/conda/lib/python3.12/site-packages/transformers/quantizers/auto.py:212: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
`low_cpu_mem_usage` was None, now default to True since model is quantized.
`low_cpu_mem_usage` was None, now default to True since model is quantized.


Reiterating the requirements:
1. Use zero-bubbling for better GPU utilization
2. Enable all FSPD2 features
3. Being torch compilable is a plus (there's great overlap anyways)
4.

Outline of approach:
1. Similar to enabling torch compile in part C, we'll patch modules and define how they should be sharded (including pre-forward/backward gathers), this is especially so for the quantized Linear4Bit module and the lora components. This is also in-line with FSDP2 requiring "bottom-up" FSDP-ification. These nested FSDP2 layers will take precedence when defining sharding behavior.
2. We'll do this iteratively, checking that losses still match. We'll also use the fact that it still ought to work on the "trivial case" of having one GPU.
3.

In [6]:
torch_compile_options = {
    "epilogue_fusion": True,
    "max_autotune": True,
    "shape_padding": True,
    "trace.enabled": True,
    "triton.cudagraphs": False,
}

# Borrow from C: we want to find and eliminate graph breaks in torch.compile
import os

os.environ["TORCHDYNAMO_VERBOSE"] = "1"
os.environ["TORCHINDUCTOR_FORCE_DISABLE_CACHES"] = "1"
os.environ["TORCHINDUCTOR_COMPILE_THREADS"] = "1"

import logging

torch._inductor.config.debug = True
torch._logging.set_logs(
    dynamo=logging.WARN,
    # inductor=logging.WARN,
    graph_breaks=True,
    # recompiles=True,
    # recompiles_verbose=True,
    # compiled_autograd_verbose=True,
)
torch._dynamo.config.verbose = False
torch._dynamo.config.suppress_errors = False

In [7]:
# Set required environment variables
os.environ["RANK"] = "0"  # Set the rank of the current process
os.environ["WORLD_SIZE"] = str(torch.cuda.device_count())  # Set the total number of processes
os.environ["MASTER_ADDR"] = "localhost"  # Set the master address
os.environ["MASTER_PORT"] = "8414"  # Set the master port

torch.distributed.init_process_group(backend="nccl")

In [8]:
# similarly, let's set configs for FSDP2

from torch.distributed.fsdp import CPUOffloadPolicy, MixedPrecisionPolicy, fully_shard
from torch.distributed import DeviceMesh

world_size = torch.distributed.get_world_size()
# world_size = torch.cuda.device_count()
rank = torch.distributed.get_rank()

device_names = [f"cuda:{i}" for i in range(world_size)]
# 1d mesh
mesh = DeviceMesh(device_type="cuda", mesh=torch.arange(world_size))

# we'll be using cpu offloading and mixed precision training
# torch.set_num_threads(os.cpu_count() // torch.cuda.device_count())

# T4s dont natively support bfloat16
# Let's use mix prec by doing gradient accumulation in float32 (later).
mp_policy = MixedPrecisionPolicy(param_dtype=torch.float16, reduce_dtype=torch.float16, cast_forward_inputs=True)

print(rank, mesh)

0 DeviceMesh('cuda', [0])


In [9]:
import triton
import triton.language as tl

@triton.jit
def dequantize_nf4_code(code_index):
    """Helper function to dequantize a 4-bit NF4 code to its float value."""
    bit3 = (code_index & 0x8) != 0  # Check if bit 3 is set (8)
    bit2 = (code_index & 0x4) != 0  # Check if bit 2 is set (4)
    bit1 = (code_index & 0x2) != 0  # Check if bit 1 is set (2)
    bit0 = (code_index & 0x1) != 0  # Check if bit 0 is set (1)

    # we avoid reading the code values directly, as they are not contiguous.
    # plus, the code values provided with linear4bit aren't exactly the same as the ones used by bnb
    # taken from bitsandbytes get_4bit_type to ensure we use the same values
    # https://github.com/bitsandbytes-foundation/bitsandbytes/blob/86b6c37a8ad448230cedb60753f63150b603a112/bitsandbytes/functional.py#L1075

    return tl.where(bit3,
        # 1xxx
        tl.where(bit2,
            # 11xx
            tl.where(bit1,
                # 111x
                tl.where(bit0, 1.0, 0.7229568362236023),
                # 110x
                tl.where(bit0, 0.5626170039176941, 0.44070982933044434)
            ),
            # 10xx
            tl.where(bit1,
                # 101x
                tl.where(bit0, 0.33791524171829224, 0.24611230194568634),
                # 100x
                tl.where(bit0, 0.16093020141124725, 0.07958029955625534)
            )
        ),
        # 0xxx
        tl.where(bit2,
            # 01xx
            tl.where(bit1,
                # 011x
                tl.where(bit0, 0.0, -0.09105003625154495),
                # 010x
                tl.where(bit0, -0.18477343022823334, -0.28444138169288635)
            ),
            # 00xx
            tl.where(bit1,
                # 001x
                tl.where(bit0, -0.39491748809814453, -0.5250730514526367),
                # 000x
                tl.where(bit0, -0.6961928009986877, -1.0)
            )
        )
    )

@triton.autotune(
    configs=[
        triton.Config({'BLOCK_SIZE': 128}),
        triton.Config({'BLOCK_SIZE': 256}),
        # triton.Config({'BLOCK_SIZE': 512}),
        # triton.Config({'BLOCK_SIZE': 1024}),
        # triton.Config({'BLOCK_SIZE': 2048}),
        # triton.Config({'BLOCK_SIZE': 4096}),
        # triton.Config({'BLOCK_SIZE': 8192}),
    ],
    key=['total_elements'],
)
@triton.jit()
def _your_dequantize_nf4_kernel(
    weight_ptr,  # [uint8]  Quantized weights, 1 byte => 2 NF4 elements
    absmax_ptr,  # [int]  One int "index" per element
    absmax2_ptr,  # [float]  One absmax2 per block
    code2_ptr,  # [float]  Absmax code lookup
    offset_ptr,  # [float]  Offset to add after absmax is decoded
    out_ptr,  # [float]  Final dequantized output
    weight_blocksize: tl.constexpr,  # [int]   Number of elements in a weight block
    absmax_blocksize: tl.constexpr, # [int]   Number of elements in a absmax block
    total_elements,  # [int]   Number of actual float elements to reconstruct
    BLOCK_SIZE: tl.constexpr,
):
    # context; we adopt the pov of a quantized byte, which corresponds to 2 elems
    pid = tl.program_id(0)
    start = pid * BLOCK_SIZE
    index = start + tl.arange(0, BLOCK_SIZE)

    offset = tl.load(offset_ptr, eviction_policy="evict_last")

    # the block index of the byte
    # weight_block_index = index // (weight_blocksize // 2) # naive approach
    # now we end up with a vector like (0, 0, ... 0, 0, 1, 1, ...)
    # we can shrink this by applying a stride of weight_blocksize // 2 so that our vector looks like (0, 1, 2, ...)
    weight_block_index = start // (weight_blocksize // 2) + tl.arange(0, BLOCK_SIZE // (weight_blocksize // 2))
    absmax_block_index = weight_block_index // absmax_blocksize # the block index of the absmax

    # mask calculations
    byte_index_mask = index < (total_elements // 2)
    weight_block_mask = weight_block_index < (total_elements // weight_blocksize)
    abs_block_mask = absmax_block_index < (total_elements // weight_blocksize // absmax_blocksize)

    # compute absmax for the block these weights belong to:
    absmax1_code = tl.load(absmax_ptr + weight_block_index, mask=weight_block_mask, eviction_policy="evict_last")
    absmax1_code = absmax1_code.to(tl.int16) # this is necessary

    # reading values from code is likely a very slow process due to lack of contiguity
    absmax1_val = tl.load(code2_ptr + absmax1_code, mask=(absmax1_code < 256))
    absmax2_val = tl.load(absmax2_ptr + absmax_block_index, mask=abs_block_mask, eviction_policy="evict_last")

    absmax = tl.fma(absmax1_val, absmax2_val, offset)

    # lookup each weight bit
    byte = tl.load(weight_ptr + index, mask=byte_index_mask, eviction_policy="evict_first")

    # matter of covention which set of bits is the first and second
    # 1 = higher bits, 2 = lower bits
    # code_index1, code_index2 = tl.inline_asm_elementwise(
    #     """
    #     // Extract low 4 bits to code_index2
    #     and.b32 $1, $2, 0x0F;
    #     // bitshift by 4
    #     shr.b32 $0, $2, 4;
    #     // Extract higher 4 bits to code_index1
    #     and.b32 $0, $0, 0x0F;
    #     """,
    #     constraints="=r,=r,r",  # 2 outputs, 1 input
    #     args=[byte],
    #     dtype=(tl.int32, tl.int32), # int32 worked best
    #     is_pure=True,
    #     pack=1
    # )

    code_index2 = byte & 0x0F  # Lower 4 bits
    code_index1 = (byte >> 4) & 0x0F  # Upper 4 bits


    # Dequantize both 4-bit values
    weight1 = dequantize_nf4_code(code_index1)[:, None].reshape(BLOCK_SIZE // (weight_blocksize // 2), (weight_blocksize // 2)) * absmax[:, None]
    weight2 = dequantize_nf4_code(code_index2)[:, None].reshape(BLOCK_SIZE // (weight_blocksize // 2), (weight_blocksize // 2)) * absmax[:, None]
    # interleaving the values here, prior to matmul, causes a slight decrease in average perf on T4 (poorer lower bound)
    # weight = dequantize_nf4_code(tl.interleave(code_index1, code_index2)).reshape(BLOCK_SIZE // (weight_blocksize // 2), weight_blocksize) * absmax[:, None]

    # combine to prevent a non-contiguous write
    weight_index = start * 2 + tl.arange(0, BLOCK_SIZE * 2)
    weight = tl.interleave(weight1, weight2)
    tl.store(out_ptr + weight_index, tl.ravel(weight), mask=weight_index < total_elements)


NF4_WEIGHT_BLOCKSIZE = 64
NF4_ABSMAX_BLOCKSIZE= 256

def triton_dequantize_nf4(weight, quant_state):
    absmax = quant_state.absmax
    shape = quant_state.shape
    offset = quant_state.offset
    state2 = quant_state.state2
    dtype = quant_state.dtype
    absmax2 = state2.absmax
    code2 = state2.code

    out = torch.empty(shape, dtype=dtype, device="cuda")
    weight_grid = lambda meta: ((out.numel() // 2 + meta["BLOCK_SIZE"] -1) // meta["BLOCK_SIZE"], )


    _your_dequantize_nf4_kernel[weight_grid](
        weight,
        absmax,
        absmax2,
        code2,
        offset,
        out,
        # these need to be constants for the purpose of
        weight_blocksize=NF4_WEIGHT_BLOCKSIZE,
        absmax_blocksize=NF4_ABSMAX_BLOCKSIZE,
        total_elements=out.numel(),
    )
    return out

In [10]:
class Linear4BitPatch(nn.Module):
    def __init__(self, bnb_linear):
        super().__init__()
        # Copy over quantized weight and metadata from the bitsandbytes layer
        # we don't need all params and features
        # don't trace param 4 bit, access its vars here

        self.weight = bnb_linear.weight.data
        self.quant_state = bnb_linear.weight.quant_state
        self.bias = bnb_linear.bias

        # Freeze weight and metadata as buffers (no grad needed)
        self.weight.requires_grad_(False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Use the triton kernel for dequantization
        W = triton_dequantize_nf4(self.weight, self.quant_state)

        # Compute linear output: x @ W^T + bias
        out = torch.matmul(x, W.t())
        if self.bias is not None:
            out = out + self.bias.to(x.dtype)
        return out

# Note: Fortunately for us, these weights are frozen, meaning the parameters won't update
# so we don't need to reshard these.



In [11]:
modules_to_replace = []
for name, module in model.named_modules():
    module_type = str(type(module))
    # we'll handle lora modules later
    if "lora" in module_type.lower():
        continue
    if "bnb" in module_type or "bitsandbytes" in module_type:
        # print(name, type(module))
        modules_to_replace.append(name)

modules_to_replace

['base_model.model.model.layers.0.self_attn.q_proj.base_layer',
 'base_model.model.model.layers.0.self_attn.k_proj.base_layer',
 'base_model.model.model.layers.0.self_attn.v_proj.base_layer',
 'base_model.model.model.layers.0.self_attn.o_proj.base_layer',
 'base_model.model.model.layers.0.mlp.gate_proj.base_layer',
 'base_model.model.model.layers.0.mlp.up_proj.base_layer',
 'base_model.model.model.layers.0.mlp.down_proj.base_layer',
 'base_model.model.model.layers.1.self_attn.q_proj.base_layer',
 'base_model.model.model.layers.1.self_attn.k_proj.base_layer',
 'base_model.model.model.layers.1.self_attn.v_proj.base_layer',
 'base_model.model.model.layers.1.self_attn.o_proj.base_layer',
 'base_model.model.model.layers.1.mlp.gate_proj.base_layer',
 'base_model.model.model.layers.1.mlp.up_proj.base_layer',
 'base_model.model.model.layers.1.mlp.down_proj.base_layer',
 'base_model.model.model.layers.2.self_attn.q_proj.base_layer',
 'base_model.model.model.layers.2.self_attn.k_proj.base_layer'

In [12]:
TARGET_BNB_MODULE_NAME = "base_model.model.model.layers.7.mlp.gate_proj.base_layer"
module = model.get_submodule(TARGET_BNB_MODULE_NAME)

INPUT_SHAPE = (4, 128, module.weight.quant_state.shape[-1]) #  (Batch, SeqLen, HiddenDim), matching in features

module

Linear4bit(in_features=4096, out_features=14336, bias=False)

In [ ]:
TEST_DTYPE = torch.float16 if not HAS_BFLOAT16 else torch.bfloat16
DEVICE = "cuda"

# --- The Test Function ---
def test_linear4bit_patch_equivalence(original_module, input_shape, dtype, device):
    print(f"--- Testing Module: {original_module} ---")
    print(f"Input shape: {input_shape}, Dtype: {dtype}, Device: {device}")

    original_module.to(device)

    # 2. Create Patched Versions (Eager and Compiled)
    # Ensure weights/state are copied *before* moving original to device if needed
    patch_eager = Linear4BitPatch(original_module).to(device)
    patch_compiled = Linear4BitPatch(original_module).to(device)

    # Explicitly compile the forward method of patch_compiled *after* instantiation
    print("Compiling Linear4BitPatch forward method...")
    try:
        # We compile the instance's method, not the class method directly
        # Note: Recompilation might happen on subsequent calls if input shapes change drastically
        patch_compiled.forward = torch.compile(
            patch_compiled.forward, # Compile the underlying function
            fullgraph=True,
            dynamic=True,
            options=torch_compile_options
        )
        print("Compilation successful (initial).")
    except Exception as e:
        print(f"ERROR: torch.compile failed during setup: {e}")
        # Decide if you want to raise e or just return
        return


    # 3. Create Sharded Version (if FSDP is enabled)
    print("Setting up sharded version...")
    # Create a *separate* instance for sharding to avoid modifying patch_compiled
    patch_sharded_compiled_base = Linear4BitPatch(original_module)
    patch_sharded_compiled_base.forward = torch.compile(
            patch_sharded_compiled_base.forward,
            fullgraph=False, dynamic=True, options=torch_compile_options
    )

    # Apply FSDP sharding
    # Use fully_shard for simplicity here, adjust if using ModuleWrapPolicy
    try:
            # Important: Shard the *module instance*
            patch_sharded_compiled = fully_shard(
                patch_sharded_compiled_base,
                mesh=mesh,
                reshard_after_forward=False, # As per your original code
                # mp_policy=mp_policy, # Pass your mixed precision policy if any
                offload_policy=CPUOffloadPolicy(pin_memory=False) # Add if needed
            )
            print("Sharding successful.")
    except Exception as e:
            print(f"ERROR: FSDP fully_shard failed during setup: {e}")
            # Continue without the sharded test if sharding fails
            patch_sharded_compiled = None


    # 4. Create Input Tensor
    x_input = torch.randn(input_shape, dtype=dtype, device=device, requires_grad=True)
    x_input_clone1 = x_input.clone().detach().requires_grad_(True)
    x_input_clone2 = x_input.clone().detach().requires_grad_(True)
    x_input_clone3 = x_input.clone().detach().requires_grad_(True)

    # --- Forward Pass ---
    print("\n--- Running Forward Pass ---")
    try:
        # do this forward pass with gradient tarcking (will do backward later)
        out_patch_eager = patch_eager(x_input_clone1)
        print("Patched Eager forward: Done")
        out_patch_compiled = patch_compiled(x_input_clone2)
        print("Patched Compiled forward: Done")
        out_patch_sharded = patch_sharded_compiled(x_input_clone3)

    except Exception as e:
        print(f"\n!!! ERROR during forward pass execution: {e}")
        # Potentially print which step failed if needed
        return # Stop the test if forward fails

    # --- Forward Pass Assertions ---
    print("\n--- Forward Pass Assertions ---")
    line_num = 0 # Placeholder for line number in assert_same if needed
    try:
        # print("Comparing: Original BnB vs Patched Eager")
        # assert_same(out_bnb, out_patch_eager, line_num, dtype)
        print("Comparing: Patched Eager vs Patched Compiled")
        assert_same(out_patch_eager, out_patch_compiled, line_num, dtype)
        print("Comparing: Patched Compiled vs Patched Sharded+Compiled")
        # Note: FSDP might alter output shape/distribution slightly depending
        # on strategy and world size. Adjust comparison logic if needed (e.g., gather first).
        # For FULL_SHARD on a single node/rank 0, output should be identical.
        assert_same(out_patch_compiled, out_patch_sharded, line_num, dtype)
        print("Forward assertions PASSED.")
    except Exception as e:
        print(f"\n!!! ERROR during forward pass assertions: {e}")
        # Potentially add more details about which comparison failed
        return # Stop if forward outputs don't match

    # --- Backward Pass ---
    print("\n--- Running Backward Pass ---")
    # Use a simple reduction for backward
    # loss_bnb = out_bnb.sum()
    loss_patch_eager = out_patch_eager.sum()
    loss_patch_compiled = out_patch_compiled.sum()
    loss_patch_sharded = out_patch_sharded.sum()

    try:
        # print("Backward: Original BnB...")
        # loss_bnb.backward()
        print("Backward: Patched Eager...")
        loss_patch_eager.backward()
        print("Backward: Patched Compiled...")
        loss_patch_compiled.backward()
        print("Backward: Patched Sharded+Compiled...")
        loss_patch_sharded.backward()
        print("Backward passes Done.")
    except Exception as e:
        print(f"\n!!! ERROR during backward pass execution: {e}")
        return # Stop the test if backward fails

    # --- Backward Pass Assertions (Input Gradients) ---
    print("\n--- Backward Pass Assertions (Input Gradients) ---")
    try:
        # grad_bnb = x_input.grad
        grad_patch_eager = x_input_clone1.grad
        grad_patch_compiled = x_input_clone2.grad
        grad_patch_sharded = x_input_clone3.grad

        # print("Comparing Grads: Original BnB vs Patched Eager")
        # assert_same(grad_bnb, grad_patch_eager, line_num, dtype)
        print("Comparing Grads: Patched Eager vs Patched Compiled")
        assert_same(grad_patch_eager, grad_patch_compiled, line_num, dtype)

        print("Comparing Grads: Patched Compiled vs Patched Sharded+Compiled")
        # Similar to forward, FSDP might affect gradient distribution.
        # Ensure comparison is fair (e.g., gradients are gathered if necessary).
        # For FULL_SHARD on single node/rank 0, should be identical.
        assert_same(grad_patch_compiled, grad_patch_sharded, line_num, dtype)

        print("Backward assertions PASSED.")
    except Exception as e:
        print(f"\n!!! ERROR during backward pass assertions: {e}")
        return

    print(f"\n--- Test for {original_module} Completed Successfully ---")


test_linear4bit_patch_equivalence(
    module,
    INPUT_SHAPE,
    TEST_DTYPE,
    DEVICE
)


--- Testing Module: Linear4bit(in_features=4096, out_features=14336, bias=False) ---
Input shape: (4, 128, 4096), Dtype: torch.bfloat16, Device: cuda
Compiling Linear4BitPatch forward method...
Compilation successful (initial).
Setting up sharded version...
Sharding successful.

--- Running Forward Pass ---
Patched Eager forward: Done
Patched Compiled forward: Done


[rank0]:W0401 08:14:42.901000 11543 site-packages/torch/_inductor/debug.py:435] [0/1] model__1_forward_4 debug trace: /workspaces/unsloth-ai-feb2025/torch_compile_debug/run_2025_04_01_08_14_23_611348-pid_11543/torchinductor/model__1_forward_4.2
[rank0]:W0401 08:14:43.085000 11543 site-packages/torch/_inductor/debug.py:435] [0/1] model__1_backward_5 debug trace: /workspaces/unsloth-ai-feb2025/torch_compile_debug/run_2025_04_01_08_14_23_611348-pid_11543/torchinductor/model__1_backward_5.3



--- Forward Pass Assertions ---
Comparing: Patched Eager vs Patched Compiled
Comparing: Patched Compiled vs Patched Sharded+Compiled
Forward assertions PASSED.

--- Running Backward Pass ---
Backward: Patched Eager...
Backward: Patched Compiled...
Backward: Patched Sharded+Compiled...
Backward passes Done.

--- Backward Pass Assertions (Input Gradients) ---
Comparing Grads: Patched Eager vs Patched Compiled
Comparing Grads: Patched Compiled vs Patched Sharded+Compiled
Backward assertions PASSED.

--- Test for Linear4bit(in_features=4096, out_features=14336, bias=False) Completed Successfully ---


In [ ]:
# patch modules for torch compile
# fully shard is an in-place op. but we'll do it later at all once
for name in modules_to_replace:
    module = model.get_submodule(name)
    # create a new patched module from the old one
    new_module = Linear4BitPatch(module)
    # compile the module too, since we have to compile before sharding
    new_module.forward = torch.compile(
        new_module.forward,
        fullgraph=False, dynamic=True, options=torch_compile_options
    )
    # Find the direct parent module
    parts = name.split('.')
    parent = model
    for i in range(len(parts) - 1):
        parent = parent.get_submodule(parts[i])

    module_name = parts[-1] # The attribute name within the parent

    setattr(parent, module_name, new_module)


Currently patched without compiling, should work

In [19]:
# shard modules
for name, module in model.named_modules():
    module_type = str(type(module))
    # we'll handle lora modules later
    if "lora" in module_type.lower():
        continue
    if "Linear4BitPatch" in module_type:
        fully_shard(module,
            reshard_after_forward=False,
            # mp_policy=mp_policy,
            mesh=mesh,
            offload_policy=CPUOffloadPolicy(pin_memory=False)
        )


# attempt to torch compile the modules
# module_names = set()
# for index, child in model.base_model.model.model.layers.named_children():
#     for name, module in child.named_children():
#         # compile the layernorm, self-attention, and mlp modules (already compiled)
#         if any(k in name for k in ["mlp"]):
#             # Add dynamic shape assumptions
#             module = torch.compile(module, fullgraph=False, dynamic=True, options=torch_compile_options)
#             setattr(child, name, module)

#         module_names.add(name)

# module_names

In [20]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        warmup_steps=1,
        max_steps=10,
        logging_steps=1,
        output_dir="outputs",
        seed=3407,
        max_seq_length=max_seq_length,
        fp16=model.get_input_embeddings().weight.dtype == torch.float16,
        bf16=model.get_input_embeddings().weight.dtype == torch.bfloat16,
        report_to="none",  # For W&B
        dataset_num_proc=4,
    ),
)

trainer.train()

torch.distributed process group is initialized, but parallel_mode != ParallelMode.DISTRIBUTED. In order to use Torch DDP, launch your script with `python -m torch.distributed.launch
torch.distributed process group is initialized, but parallel_mode != ParallelMode.DISTRIBUTED. In order to use Torch DDP, launch your script with `python -m torch.distributed.launch
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
`use_cache=True` is incompatibl

Step,Training Loss
1,1.406200
2,2.178800
3,2.001600
4,3.252400
5,1.953800
6,2.754800
7,1.932000
8,1.435900
9,1.888100
10,2.098000


TrainOutput(global_step=10, training_loss=2.0901565551757812, metrics={'train_runtime': 72.4001, 'train_samples_per_second': 0.276, 'train_steps_per_second': 0.138, 'total_flos': 7376122994688.0, 'train_loss': 2.0901565551757812})